# JetBot HRI — buổi chạy đầu qua Jupyter

Notebook này chạy **trên JetBot** sau khi upload và giải nén `jetbot_hri_upload.zip`. Mở file từ thư mục `~/jetbot_hri` trong Jupyter. Các cell dưới đây chỉ kiểm tra môi trường, WiFi, pin mock, dry-run FSM và tests. **Không cell nào điều khiển motor thật.**

Trước khi chạy: tắt kernel các notebook JetBot khác nếu chúng đang dùng camera hoặc motor. Chạy lần lượt từ trên xuống. Nếu cell báo lỗi, dừng và lưu output để kiểm tra.

In [ ]:
import os
import sys
from pathlib import Path
import subprocess

ROOT = Path.cwd()
print('Python kernel:', sys.executable)
print('Thư mục:', ROOT)
assert (ROOT / 'setup.py').is_file(), 'Mở notebook từ thư mục ~/jetbot_hri, nơi có setup.py'
print('Source OK')

## 1. Kiểm tra JetPack, Python và thư viện

Nếu bản Jupyter có **New → Terminal**, chạy thêm `bash scripts/inspect_platform.sh | tee platform.txt` ở đó. Không nâng cấp Torch/TensorRT trước khi lưu kết quả inspect.

In [ ]:
checks = [
    ['cat', '/etc/nv_tegra_release'],
    ['python3', '--version'],
    [sys.executable, '-m', 'pip', 'show', 'jetbot'],
    [sys.executable, '-m', 'pip', 'show', 'PyYAML'],
    ['hostname', '-I'],
    ['nmcli', 'device', 'status'],
]
for command in checks:
    print('\n$', ' '.join(command))
    try:
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                universal_newlines=True, timeout=15)
        print(result.stdout.strip() or '(không có output)')
    except Exception as exc:
        print(type(exc).__name__ + ':', exc)

## 2. Cài source vào chính Python kernel này

Cell này chỉ cài package JetBot HRI và dependency nhẹ của nó. Nếu lỗi cài đặt, dừng lại và gửi output; không thử cài lại PyTorch/TensorRT tùy ý. Sau khi cài xong, nếu import chưa được, khởi động lại kernel rồi chạy lại từ đầu.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', '-e', str(ROOT)],
                        cwd=str(ROOT), timeout=180)
assert result.returncode == 0, 'Cài source thất bại; lưu output của cell này'

## 3. Dry-run toàn FSM và unit tests

Dry-run dùng observation và motor mock; kết quả mong đợi là `Dry-run COMPLETE`. Nó **chưa chứng minh** camera, gesture, pin hay điều khiển motor thật hoạt động.

In [ ]:
result = subprocess.run([sys.executable, '-B', '-m', 'jetbot_hri.main',
                         '--mode', 'dry-run', '--log', 'logs/jupyter-dry-run.jsonl'],
                        cwd=str(ROOT), timeout=30)
assert result.returncode == 0, 'Dry-run lỗi; lưu output và không chạy bước tiếp theo'

In [ ]:
result = subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover',
                         '-s', 'tests', '-q'], cwd=str(ROOT), timeout=60)
assert result.returncode == 0, 'Có test lỗi; lưu output để kiểm tra'

## 4. Kiểm tra WiFi và pin mock

Không truy cập INA219 hay motor thật. Nếu WiFi reader báo `nmcli` không có, gửi lại output để điều chỉnh theo image JetBot.

In [ ]:
for module, extra in [('scripts.test_wifi', []), ('scripts.test_battery', ['--mock'])]:
    print('\n===', module, '===')
    result = subprocess.run([sys.executable, '-B', '-m', module] + extra,
                            cwd=str(ROOT), timeout=30)
    print('Exit code:', result.returncode)

## Bước tiếp theo

Dừng ở đây và gửi output môi trường + dry-run cho nhóm. Sau đó kiểm tra camera riêng bằng `python3 -m scripts.test_camera` trong Terminal khi không notebook nào đang giữ camera. `--mode perception` hiện cần weights pose **và** collision; `--mode robot` còn cần calibration thực tế và kiểm tra safety. Không chạy hai mode đó hoặc bật service tự khởi động ở buổi đầu. Xem `JETBOT_JUPYTER_FIRST_RUN.md` cho kế hoạch dataset `FREE/BLOCKED` và trình tự thử nghiệm.